In [1]:
# ============================================================
# CELL 1 — FIX LỖI TORCH: Chỉ cần chạy cell này, KHÔNG làm gì khác
# ============================================================
# Lỗi "module functions cannot set METH_CLASS or METH_STATIC"
# = pip install trong session hiện tại đã corrupt torch trong RAM.
# Kernel cần được reset hoàn toàn trước khi import torch.
#
# HƯỚNG DẪN:
#   Bước 1: Trên thanh menu Kaggle, bấm  Run > Restart & Clear Output
#   Bước 2: Sau khi kernel restart xong (status = Idle), chạy Cell 2 ngay
#   Bước 3: KHÔNG chạy pip install nào trước Cell 2
#
# Tại sao? Kaggle đã có sẵn torch, transformers, boto3, scikit-learn.
# Việc pip install trong cùng session làm thay đổi shared libs đang
# được nạp vào RAM → torch._C crash khi import lần 2.

import sys
print(f"Python: {sys.version}")

try:
    import torch
    print(f"✅ Torch {torch.__version__} hoạt động bình thường!")
    print(f"   CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
except Exception as e:
    print(f"❌ Torch vẫn lỗi: {e}")
    print("→ Thử bấm: Run > Factory Reset Session (nếu có) hoặc tạo notebook mới")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
✅ Torch 2.10.0+cu128 hoạt động bình thường!
   CUDA available: True
   GPU: Tesla T4


In [ ]:
# ============================================================
# CELL 2: SPAMSHIELD AI — FULL TRAINING PIPELINE (NÂNG CẤP)
# ============================================================

import os, re, json, glob, time, tarfile
import torch
import boto3
import pandas as pd
import torch.nn as nn
import numpy as np
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup   # ← tốt hơn linear cho fine-tune
)
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from kaggle_secrets import UserSecretsClient

# ============================================================
# 1. CẤU HÌNH
# ============================================================
user_secrets = UserSecretsClient()
os.environ['AWS_ACCESS_KEY_ID']     = user_secrets.get_secret("AWS_ACCESS_KEY_ID")
os.environ['AWS_SECRET_ACCESS_KEY'] = user_secrets.get_secret("AWS_SECRET_ACCESS_KEY")
os.environ['AWS_DEFAULT_REGION']    = user_secrets.get_secret("AWS_DEFAULT_REGION")
os.environ['S3_BUCKET_NAME']        = user_secrets.get_secret("S3_BUCKET_NAME")

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "vinai/phobert-base-v2"   # PhoBERT — tối ưu cho tiếng Việt

# --- HYPERPARAMS CHO PHOBERT ---
# PhoBERT: hidden=768, max_position=258 (thực dùng tối đa 256 tokens)
# Tokenizer BPE trên văn bản tiếng Việt → MAX_LEN=128 là đủ & tiết kiệm VRAM
MAX_LEN      = 128    # PhoBERT hoạt động tốt nhất ở 128
BATCH_SIZE   = 16     # PhoBERT nhỏ hơn DeBERTa → fit batch 16 trên T4
ACCUM_STEPS  = 2      # Effective batch = 16 × 2 = 32 ✓
EPOCHS       = 6
LR           = 2e-5   # LR chuẩn cho RoBERTa-base
WARMUP_RATIO = 0.1
LABEL_SMOOTH = 0.1
PATIENCE     = 2

print(f"🖥️  Device: {DEVICE}")
print(f"📦  Model: {MODEL_NAME}")

# ============================================================
# 2. LOAD & TIỀN XỬ LÝ
# ============================================================
def load_teencode(path):
    with open(path, encoding='utf-8') as f:
        raw = json.load(f)
    flat = {}
    for k, v in raw.items():
        if not k.startswith('_') and isinstance(v, dict):
            flat.update(v)
    return flat

def clean_text_and_extract_features(text, teencode_map):
    text = str(text).lower()
    # Trích đặc trưng TRƯỚC khi xóa ký tự đặc biệt
    contains_url   = 1.0 if re.search(r'https?://\S+', text) else 0.0
    contains_phone = 1.0 if re.search(r'(0\d{9,10})', text) else 0.0
    # Thay teencode
    for pattern, replacement in teencode_map.items():
        text = re.sub(pattern, replacement, text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text, contains_url, contains_phone

# Auto-detect paths
csv_paths  = glob.glob('/kaggle/input/**/cleaned_dataset.csv', recursive=True)
json_paths = glob.glob('/kaggle/input/**/Teencode_map.json', recursive=True)
if not csv_paths or not json_paths:
    raise FileNotFoundError("❌ Không tìm thấy file! Kiểm tra lại Input dataset.")

DATA_PATH     = csv_paths[0]
TEENCODE_PATH = json_paths[0]
print(f"✅ Data: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
teencode_map = load_teencode(TEENCODE_PATH)

feats = df['content'].apply(lambda x: clean_text_and_extract_features(x, teencode_map))
df['clean_text']    = [f[0] for f in feats]
df['contains_url']  = [f[1] for f in feats]
df['contains_phone'] = [f[2] for f in feats]

# Chuẩn hoá label
df['label'] = df['label'].astype(str).str.lower().str.strip()
df = df[df['label'].isin(['ham', 'spam', 'scam'])].copy()
label_map = {'ham': 0, 'spam': 1, 'scam': 2}
df['label_idx'] = df['label'].map(label_map)
print(f"📊 Phân bố nhãn:\n{df['label'].value_counts()}")

# Split 70 / 20 / 10
train_df, temp_df = train_test_split(df, test_size=0.3,   stratify=df['label_idx'], random_state=42)
val_df,  test_df  = train_test_split(temp_df, test_size=0.3333, stratify=temp_df['label_idx'], random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ============================================================
# 3. DATASET
# ============================================================
class SpamShieldDataset(Dataset):
    def __init__(self, texts, urls, phones, labels, tokenizer, max_len=MAX_LEN):
        self.texts, self.urls, self.phones, self.labels = texts, urls, phones, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i],
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'extra_features': torch.tensor([self.urls[i], self.phones[i]], dtype=torch.float),
            'labels':         torch.tensor(self.labels[i], dtype=torch.long)
        }

# ============================================================
# 4. MODEL — KIẾN TRÚC CHUẨN (đồng nhất train & inference)
# ============================================================
class PhoBERTCustom(nn.Module):
    """
    PhoBERT-base-v2: [CLS] embedding (768) + 2 extra features → Linear → 3 classes
    Dropout nhất quán giữa train và inference để load_state_dict không lỗi.
    """
    def __init__(self, model_name=MODEL_NAME, num_classes=3):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(model_name)
        self.drop       = nn.Dropout(p=0.3)
        hidden_size     = self.bert.config.hidden_size
        self.classifier = nn.Linear(hidden_size + 2, num_classes)

    def forward(self, input_ids, attention_mask, extra_features):
        out          = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled       = out.last_hidden_state[:, 0]          # [CLS]
        pooled       = self.drop(pooled)
        concat       = torch.cat((pooled, extra_features), dim=1)
        return self.classifier(concat)

# ============================================================
# 5. LABEL SMOOTHING LOSS
# ============================================================
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=LABEL_SMOOTH, num_classes=3):
        super().__init__()
        self.smoothing   = smoothing
        self.num_classes = num_classes

    def forward(self, pred, target):
        confidence = 1.0 - self.smoothing
        smooth_val = self.smoothing / (self.num_classes - 1)
        one_hot    = torch.full_like(pred, smooth_val)
        one_hot.scatter_(1, target.unsqueeze(1), confidence)
        log_prob   = torch.nn.functional.log_softmax(pred, dim=1)
        return -(one_hot * log_prob).sum(dim=1).mean()

# ============================================================
# 6. KHỞI TẠO
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)  # PhoBERT cần use_fast=False
model     = PhoBERTCustom(MODEL_NAME, num_classes=3).to(DEVICE)

def make_loader(df_split, shuffle=False):
    ds = SpamShieldDataset(
        df_split.clean_text.to_numpy(),
        df_split.contains_url.to_numpy(),
        df_split.contains_phone.to_numpy(),
        df_split.label_idx.to_numpy(),
        tokenizer
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=True)

train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df)
test_loader  = make_loader(test_df)

total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

# Phân biệt LR: backbone nhỏ hơn, classifier lớn hơn (discriminative LR)
optimizer = AdamW([
    {'params': model.bert.parameters(),       'lr': LR},
    {'params': model.classifier.parameters(), 'lr': LR * 5},
    {'params': model.drop.parameters(),       'lr': LR},
], weight_decay=0.01)

scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn   = LabelSmoothingCrossEntropy().to(DEVICE)

# PhoBERT dùng FP32 embedding → AMP fp16 hoạt động bình thường, không bị lỗi
USE_BF16 = False
USE_AMP  = torch.cuda.is_available()   # Bật fp16 AMP → tăng tốc ~1.5-2× trên T4
scaler   = torch.amp.GradScaler('cuda', enabled=USE_AMP)
print(f"⚙️  Precision: {'fp16 AMP' if USE_AMP else 'fp32'}")

# ============================================================
# 7. TRAIN LOOP
# ============================================================
def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            mask  = batch['attention_mask'].to(DEVICE)
            extra = batch['extra_features'].to(DEVICE)
            labs  = batch['labels']
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(ids, mask, extra)
            preds = logits.argmax(dim=1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labs.numpy())
    return f1_score(all_labels, all_preds, average='macro'), all_preds, all_labels

best_val_f1  = 0.0
patience_cnt = 0
history      = []

print(f"\n🚀 Bắt đầu huấn luyện | {EPOCHS} epochs | Effective batch={BATCH_SIZE*ACCUM_STEPS}")
print("="*60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        extra = batch['extra_features'].to(DEVICE)
        labs  = batch['labels'].to(DEVICE)

        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(ids, mask, extra)
            loss   = loss_fn(logits, labs) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS

    avg_loss = total_loss / len(train_loader)
    val_f1, _, _ = evaluate(val_loader)
    elapsed = time.time() - t0

    history.append({'epoch': epoch, 'loss': avg_loss, 'val_f1': val_f1})
    print(f"Epoch {epoch}/{EPOCHS} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | ⏱ {elapsed:.0f}s")

    # Lưu best model
    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        patience_cnt = 0
        torch.save(model.state_dict(), 'model.pt')
        print(f"  💾 Saved best model (Val F1={val_f1:.4f})")
    else:
        patience_cnt += 1
        print(f"  ⏳ No improvement ({patience_cnt}/{PATIENCE})")
        if patience_cnt >= PATIENCE:
            print("  🛑 Early stopping triggered!")
            break

# ============================================================
# 8. ĐÁNH GIÁ TRÊN TEST SET
# ============================================================
print("\n" + "="*60)
print("📊 ĐÁNH GIÁ TRÊN TEST SET (best model)")
model.load_state_dict(torch.load('model.pt', map_location=DEVICE))
_, test_preds, test_labels = evaluate(test_loader)
print(classification_report(test_labels, test_preds,
                             target_names=['ham', 'spam', 'scam'], digits=4))

# ============================================================
# 9. LƯU TOKENIZER
# ============================================================
tokenizer.save_pretrained('./tokenizer')
print("✅ Đã lưu model.pt + tokenizer/")

🖥️  Device: cuda
📦  Model: vinai/phobert-base-v2
✅ Data: /kaggle/input/datasets/khoiho1234/cleaned-data-csv/cleaned_dataset.csv
📊 Phân bố nhãn:
label
scam    1816
spam    1784
ham     1765
Name: count, dtype: int64
Train: 3755 | Val: 1073 | Test: 537


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


⚙️  Precision: fp16 AMP

🚀 Bắt đầu huấn luyện | 6 epochs | Effective batch=32
Epoch 1/6 | Loss: 0.7443 | Val F1: 0.9459 | ⏱ 29s
  💾 Saved best model (Val F1=0.9459)
Epoch 2/6 | Loss: 0.4931 | Val F1: 0.9483 | ⏱ 29s
  💾 Saved best model (Val F1=0.9483)
Epoch 3/6 | Loss: 0.4539 | Val F1: 0.9655 | ⏱ 29s
  💾 Saved best model (Val F1=0.9655)


In [ ]:
# ============================================================
# CELL 3: BUNDLE PHOBERT LOCAL + INFERENCE.PY + ĐÓNG GÓI + ĐẨY S3
# FIX: thêm requirements.txt → container tự cài transformers khi load
# ============================================================
import os, re, tarfile, glob, boto3
from transformers import AutoTokenizer, AutoModel, AutoConfig
 
# ---- 3A: Download & lưu PhoBERT + tokenizer hoàn toàn local ----
print("📥 Saving PhoBERT base model locally (chỉ cần chạy 1 lần)...")
os.makedirs("phobert_base", exist_ok=True)
os.makedirs("tokenizer",    exist_ok=True)
 
base_model = AutoModel.from_pretrained("vinai/phobert-base-v2")
base_model.save_pretrained("phobert_base")
 
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2", use_fast=False)
tokenizer.save_pretrained("tokenizer")
print("✅ phobert_base/ và tokenizer/ đã lưu xong")
 
# ---- 3B: requirements.txt — FIX CHÍNH: container sẽ pip install trước khi load model ----
with open("code/requirements.txt", "w") as f:
    f.write("transformers==4.35.2\n")
    f.write("sentencepiece==0.1.99\n")
    f.write("protobuf==3.20.3\n")
print("✅ code/requirements.txt OK  ← đây là fix cho lỗi 'No module named transformers'")
 
# ---- 3C: Ghi inference.py — load HOÀN TOÀN từ local, không cần internet ----
inference_code = '''import json, re, os, torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, AutoConfig
 
_tokenizer = None
_device    = None
 
class PhoBERTCustom(nn.Module):
    def __init__(self, config_path, num_classes=3):
        super().__init__()
        config          = AutoConfig.from_pretrained(config_path)
        self.bert       = AutoModel.from_config(config)
        self.drop       = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(config.hidden_size + 2, num_classes)
 
    def forward(self, input_ids, attention_mask, extra_features):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]
        pooled = self.drop(pooled)
        concat = torch.cat((pooled, extra_features), dim=1)
        return self.classifier(concat)
 
def model_fn(model_dir):
    global _tokenizer, _device
    _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[SpamShield] Device: {_device}")
 
    tokenizer_path = os.path.join(model_dir, "tokenizer")
    phobert_path   = os.path.join(model_dir, "phobert_base")
 
    print("[SpamShield] Loading tokenizer...")
    _tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=False)
    print("[SpamShield] Tokenizer OK")
 
    print("[SpamShield] Building model from local config...")
    model = PhoBERTCustom(phobert_path, num_classes=3)
 
    weights_path = os.path.join(model_dir, "model.pt")
    print(f"[SpamShield] Loading weights from {weights_path}...")
    model.load_state_dict(torch.load(weights_path, map_location=_device))
    model.to(_device).eval()
    print("[SpamShield] Model ready!")
    return model
 
def input_fn(request_body, content_type="application/json"):
    return json.loads(request_body)
 
def predict_fn(input_data, model):
    text           = input_data.get("inputs", "")
    contains_url   = 1.0 if re.search(r"https?://\\S+", text) else 0.0
    contains_phone = 1.0 if re.search(r"(0\\d{9,10})", text) else 0.0
 
    enc   = _tokenizer(text, max_length=128, padding="max_length",
                       truncation=True, return_tensors="pt")
    extra = torch.tensor([[contains_url, contains_phone]], dtype=torch.float)
 
    with torch.no_grad():
        logits = model(
            enc["input_ids"].to(_device),
            enc["attention_mask"].to(_device),
            extra.to(_device)
        )
        probs = torch.softmax(logits, dim=1)
 
    label_names = ["ham", "spam", "scam"]
    pred_idx    = probs.argmax(dim=1).item()
    return {
        "prediction": label_names[pred_idx],
        "probabilities": {
            name: round(float(p), 4)
            for name, p in zip(label_names, probs[0].cpu().tolist())
        }
    }
 
def output_fn(prediction, accept="application/json"):
    return json.dumps(prediction), accept
'''
 
os.makedirs("code", exist_ok=True)
with open("code/inference.py", "w", encoding="utf-8") as f:
    f.write(inference_code.strip())
print("✅ code/inference.py OK")
 
# ---- 3D: config.properties ----
with open("config.properties", "w") as f:
    f.write("default_response_timeout=300\n")
    f.write("unregister_model_timeout=300\n")
    f.write("disable_system_metrics=true\n")
print("✅ config.properties OK")
 
# ---- 3E: Tìm Teencode_map ----
json_paths   = glob.glob('/kaggle/input/**/Teencode_map.json', recursive=True)
teencode_src = json_paths[0] if json_paths else None
if teencode_src:
    print(f"✅ Teencode map: {teencode_src}")
else:
    print("⚠️  Không tìm thấy Teencode_map.json — bỏ qua")
 
# ---- 3F: Đóng gói model.tar.gz ----
print("\n📦 Đang đóng gói model.tar.gz...")
with tarfile.open('model.tar.gz', 'w:gz') as tar:
    tar.add('model.pt')
    tar.add('code/',         arcname='code')   # bao gồm cả requirements.txt + inference.py
    tar.add('tokenizer/',    arcname='tokenizer')
    tar.add('phobert_base/', arcname='phobert_base')
    tar.add('config.properties')
    if teencode_src:
        tar.add(teencode_src, arcname='Teencode_map.json')
 
size_mb = os.path.getsize('model.tar.gz') / 1024 / 1024
print(f"✅ Đóng gói xong! ({size_mb:.1f} MB)")
 
# ---- 3G: Verify tar chứa đúng requirements.txt ----
print("\n🔍 Kiểm tra nội dung tar.gz:")
with tarfile.open('model.tar.gz', 'r:gz') as tar:
    members = tar.getnames()
    for m in members:
        print(f"  {m}")
assert 'code/requirements.txt' in members, "❌ THIẾU requirements.txt trong tar!"
assert 'code/inference.py'     in members, "❌ THIẾU inference.py trong tar!"
print("✅ Tar hợp lệ, có requirements.txt")
 
# ---- 3H: Upload lên S3 ----
S3_BUCKET = os.environ.get('S3_BUCKET_NAME', 'spam-detection-doannhom')
S3_KEY    = 'models/videberta/model.tar.gz'
REGION    = os.environ.get('AWS_DEFAULT_REGION', 'ap-southeast-1')
 
print(f"\n🚀 Uploading lên s3://{S3_BUCKET}/{S3_KEY} ...")
s3        = boto3.client('s3', region_name=REGION)
file_size = os.path.getsize('model.tar.gz')
uploaded  = {'bytes': 0}
 
def progress(chunk):
    uploaded['bytes'] += chunk
    print(f"\r  Upload: {uploaded['bytes']/file_size*100:.1f}%", end='', flush=True)
 
s3.upload_file('model.tar.gz', S3_BUCKET, S3_KEY, Callback=progress)
print(f"\n✅ Upload xong! s3://{S3_BUCKET}/{S3_KEY}")

In [ ]:
# ============================================================
# CELL 4: XÓA ENDPOINT CŨ + TẠO LẠI ENDPOINT MỚI (AUTO, KHÔNG CẦN THAO TÁC THỦ CÔNG)
# ============================================================
import boto3, time
 
REGION          = 'ap-southeast-1'
ENDPOINT_NAME   = 'spam-detection-endpoint-final'
S3_MODEL_URI    = f's3://{S3_BUCKET}/{S3_KEY}'
ROLE_ARN        = 'arn:aws:iam::992409270804:role/SageMakerExecutionRole'
CONTAINER_IMAGE = '763104351884.dkr.ecr.ap-southeast-1.amazonaws.com/pytorch-inference:1.13.1-gpu-py39-cu117-ubuntu20.04-sagemaker'
INSTANCE_TYPE   = 'ml.g4dn.xlarge'
 
sm = boto3.client('sagemaker', region_name=REGION)
 
# ---- 4A: Xóa endpoint / endpoint config / model nếu tồn tại ----
print("🗑️  Dọn dẹp tài nguyên SageMaker cũ...")
 
def safe_delete(fn, label, **kwargs):
    try:
        fn(**kwargs)
        print(f"  ✅ Đã xóa: {label}")
    except sm.exceptions.ClientError as e:
        code = e.response['Error']['Code']
        if code in ('ValidationException', 'ResourceNotFoundException'):
            print(f"  ℹ️  Không tồn tại (bỏ qua): {label}")
        else:
            raise
 
# Phải xóa endpoint trước, rồi mới xóa config & model
safe_delete(sm.delete_endpoint,        "Endpoint",        EndpointName=ENDPOINT_NAME)
 
# Chờ endpoint xóa xong (tối đa 5 phút)
print("  ⏳ Chờ endpoint xóa hoàn toàn...")
for _ in range(30):
    try:
        sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        time.sleep(10)
        print("    ...", end='', flush=True)
    except sm.exceptions.ClientError:
        print("\n  ✅ Endpoint đã xóa xong")
        break
 
safe_delete(sm.delete_endpoint_config, "EndpointConfig", EndpointConfigName=ENDPOINT_NAME)
safe_delete(sm.delete_model,           "Model",          ModelName=ENDPOINT_NAME)
 
print("✅ Dọn dẹp xong!\n")
 
# ---- 4B: Tạo Model mới ----
print(f"🔧 Tạo SageMaker Model: {ENDPOINT_NAME}")
sm.create_model(
    ModelName        = ENDPOINT_NAME,
    ExecutionRoleArn = ROLE_ARN,
    PrimaryContainer = {
        'Image':        CONTAINER_IMAGE,
        'ModelDataUrl': S3_MODEL_URI,
        'Environment': {
            'SAGEMAKER_PROGRAM':         'inference.py',
            'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml/model/code',
        }
    }
)
print("✅ Model tạo xong")
 
# ---- 4C: Tạo Endpoint Config ----
print(f"🔧 Tạo EndpointConfig: {ENDPOINT_NAME}")
sm.create_endpoint_config(
    EndpointConfigName = ENDPOINT_NAME,
    ProductionVariants = [{
        'VariantName':         'AllTraffic',
        'ModelName':           ENDPOINT_NAME,
        'InstanceType':        INSTANCE_TYPE,
        'InitialInstanceCount': 1,
    }]
)
print("✅ EndpointConfig tạo xong")
 
# ---- 4D: Tạo Endpoint ----
print(f"🚀 Tạo Endpoint: {ENDPOINT_NAME}")
sm.create_endpoint(
    EndpointName       = ENDPOINT_NAME,
    EndpointConfigName = ENDPOINT_NAME,
)
print("✅ Endpoint đang Creating...\n")
 
# ---- 4E: Poll trạng thái cho đến InService (hoặc Failed) ----
print("⏳ Chờ endpoint InService (thường mất 8–12 phút)...")
start = time.time()
last_status = None
 
while True:
    resp   = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = resp['EndpointStatus']
    elapsed = int(time.time() - start)
 
    if status != last_status:
        print(f"\n  [{elapsed:>4}s] Status: {status}")
        last_status = status
    else:
        print(f"\r  [{elapsed:>4}s] {status}...", end='', flush=True)
 
    if status == 'InService':
        print(f"\n\n🎉 ENDPOINT SẴN SÀNG sau {elapsed}s!")
        print(f"   Endpoint: {ENDPOINT_NAME}")
        print("   Bây giờ có thể test Lambda hoặc gọi trực tiếp từ extension.")
        break
 
    if status in ('Failed', 'RollingBack'):
        reason = resp.get('FailureReason', 'Không rõ')
        print(f"\n\n❌ ENDPOINT FAILED!")
        print(f"   Lý do: {reason}")
        print("   → Kiểm tra CloudWatch Logs để xem chi tiết lỗi inference.py")
        break
 
    time.sleep(15)
 